# Implementing an API using PurpleAir

### 1.Imports

In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import requests
import os
from dotenv import load_dotenv
from datetime import datetime
import folium

### 2. Setting up the API request

In [8]:
load_dotenv()

API_KEY = os.getenv("PURPLEAIR_READ_KEY")
BASE_URL = "https://api.purpleair.com/v1"

headers = {
    "X-API-Key": API_KEY
}

#Get a single sensor's data
sensor_index = 131075  #Replace with your target sensor's index if you have one

response = requests.get(
    f"{BASE_URL}/sensors/{sensor_index}",
    headers=headers,
    params={"fields": "name ,pm2.5, humidity, temperature"}
)

if response.status_code == 200:
    data = response.json()
    print(data)
else:
    print(f"Error {response.status_code}:", response.text)

{'api_version': 'V1.2.0-1.1.45', 'time_stamp': 1775785420, 'data_time_stamp': 1775785411, 'sensor': {'sensor_index': 131075, 'name': 'Mariners Bluff', 'humidity': 100, 'temperature': 83, 'pm2.5': 14.9}}


In [9]:
#Responses for KY general area, as this will end up being a box and include data from a few surrounding states
response = requests.get(
    "https://api.purpleair.com/v1/sensors",
    headers=headers,
    params={
        "fields": "name, pm2.5, humidity, temperature, latitude, longitude",
        "location_type": 0,
        "nwlng": -89.57,
        "nwlat": 39.15,
        "selng": -81.96,
        "selat": 36.49,
    }
)

data = response.json()
print(f"Found {len(data['data'])} sensors")
print(data)

Found 120 sensors
{'api_version': 'V1.2.0-1.1.45', 'time_stamp': 1775785420, 'data_time_stamp': 1775785385, 'location_type': 0, 'max_age': 604800, 'firmware_default_version': '7.02', 'fields': ['sensor_index', 'name', 'latitude', 'longitude', 'humidity', 'temperature', 'pm2.5'], 'data': [[262243, 'TDEC APC - Kingsport - 262243', 36.538925, -82.52163, 37, 64, 13.4], [263785, 'EWV Whitman 1', 37.79283, -82.030815, 55, 61, 17.2], [263797, 'EWV Logan 2', 37.904606, -82.10735, 45, 64, 16.7], [263809, 'EWV Point Pleasant 3', 38.87221, -82.12882, 40, 69, 14.2], [273763, 'Boone Block', 39.08691, -84.50917, 32, 75, 13.5], [13031, 'Mount Vernon,Ind.', 37.94211, -87.89055, 25, 82, 19.2], [276356, 'Floyd Street Old Louisville', 38.23082, -85.75195, 25, 81, 15.7], [278523, 'EWV Five Mile 1', 37.742935, -82.14651, 58, 60, 12.7], [278531, 'EWV Logan 3', 37.881184, -82.081055, 36, 67, 15.6], [278566, 'EWV Point Pleasant 1', 38.91047, -82.11349, 47, 67, 24.4], [279406, 'Barbourville, KY / Knox County',

In [10]:
fields = data['fields']
sensors = data['data']

for sensor in sensors:
    sensor_dict = dict(zip(fields, sensor))
    print(f"Name: {sensor_dict['name']}")
    print(f"  PM2.5:       {sensor_dict['pm2.5']}")
    print(f"  Humidity:    {sensor_dict['humidity']}")
    print(f"  Temperature: {sensor_dict['temperature']}")
    print()

Name: TDEC APC - Kingsport - 262243
  PM2.5:       13.4
  Humidity:    37
  Temperature: 64

Name: EWV Whitman 1
  PM2.5:       17.2
  Humidity:    55
  Temperature: 61

Name: EWV Logan 2
  PM2.5:       16.7
  Humidity:    45
  Temperature: 64

Name: EWV Point Pleasant 3
  PM2.5:       14.2
  Humidity:    40
  Temperature: 69

Name: Boone Block
  PM2.5:       13.5
  Humidity:    32
  Temperature: 75

Name: Mount Vernon,Ind.
  PM2.5:       19.2
  Humidity:    25
  Temperature: 82

Name: Floyd Street Old Louisville
  PM2.5:       15.7
  Humidity:    25
  Temperature: 81

Name: EWV Five Mile 1
  PM2.5:       12.7
  Humidity:    58
  Temperature: 60

Name: EWV Logan 3
  PM2.5:       15.6
  Humidity:    36
  Temperature: 67

Name: EWV Point Pleasant 1
  PM2.5:       24.4
  Humidity:    47
  Temperature: 67

Name: Barbourville, KY / Knox County
  PM2.5:       14.9
  Humidity:    30
  Temperature: 72

Name: Cincinnati State
  PM2.5:       23.8
  Humidity:    35
  Temperature: 72

Name: Cincy 

### 3. Interactive Visualization for API call

In [11]:
#Center map on Kentucky
m = folium.Map(location=[37.8, -85.8], zoom_start=7)

for sensor in sensors:
    sensor_dict = dict(zip(fields, sensor))
    
    folium.Marker(
        location=[sensor_dict['latitude'], sensor_dict['longitude']],
        popup=folium.Popup(
            f"<b>{sensor_dict['name']}</b><br>"
            f"PM2.5: {sensor_dict['pm2.5']}<br>"
            f"Humidity: {sensor_dict['humidity']}<br>"
            f"Temperature: {sensor_dict['temperature']}",
            max_width=200
        ),
        tooltip=sensor_dict['name']
    ).add_to(m)

m.save("../End_Visualizations/kentucky_sensors_map.html")
print("Map saved.")

Map saved.


### 4. A function that finds sensors based on the area's air quality

In [12]:
def get_sensors_by_air_quality(threshold):
    """
    Returns sensors where PM2.5 exceeds the given threshold.
    
    Parameters:
        threshold (float): PM2.5 value to filter by
    
    Returns:
        list of dicts with sensor data
    """
    response = requests.get(
        "https://api.purpleair.com/v1/sensors",
        headers=headers,
        params={
            "fields": "name,pm2.5,humidity,temperature,latitude,longitude",
            "location_type": 0,
            "nwlng": -89.57,
            "nwlat": 39.15,
            "selng": -81.96,
            "selat": 36.49,
        }
    )

    data = response.json()
    fields = data['fields']
    sensors = data['data']

    results = []
    for sensor in sensors:
        sensor_dict = dict(zip(fields, sensor))
        if sensor_dict['pm2.5'] is not None and sensor_dict['pm2.5'] > threshold:
            results.append(sensor_dict)

    print(f"Found {len(results)} sensors above PM2.5 threshold of {threshold}")
    return results

#Example usage — EPA's "Good" air quality limit is 12.0
flagged_sensors = get_sensors_by_air_quality(12.0)
for sensor in flagged_sensors:
    print(f"Name: {sensor['name']}, PM2.5: {sensor['pm2.5']}")

Found 103 sensors above PM2.5 threshold of 12.0
Name: TDEC APC - Kingsport - 262243, PM2.5: 13.4
Name: EWV Whitman 1, PM2.5: 17.2
Name: EWV Logan 2, PM2.5: 16.7
Name: EWV Point Pleasant 3, PM2.5: 14.2
Name: Boone Block, PM2.5: 13.5
Name: Mount Vernon,Ind., PM2.5: 19.2
Name: Floyd Street Old Louisville, PM2.5: 15.7
Name: EWV Five Mile 1, PM2.5: 12.7
Name: EWV Logan 3, PM2.5: 15.6
Name: EWV Point Pleasant 1, PM2.5: 24.4
Name: Barbourville, KY / Knox County, PM2.5: 14.9
Name: Cincinnati State, PM2.5: 23.8
Name: Cincy Air Watch- Zoo, PM2.5: 21.7
Name: Cincy Air Watch- Citylink Center, PM2.5: 20.3
Name: OVSA TC River View, PM2.5: 16.3
Name: OVSA- Sisters, PM2.5: 14.2
Name: OVSA - Ferdinand Fire Tower, PM2.5: 14.6
Name: OVSA- Enterprise Camp, PM2.5: 12.2
Name: OVSA-Franklin, PM2.5: 16.3
Name: OVSA-West Willow Road, PM2.5: 15.4
Name: OVSAKATIE, PM2.5: 17.4
Name: OVSA- Garvin Park, PM2.5: 24.1
Name: OVSA- Lilydale, PM2.5: 17.6
Name: Limerick, PM2.5: 16.4
Name: OVSA/Old Huntingburg Road, PM2.5:

### 5. A function that classifies AQI risk for those unfamiliar with the number system

In [ ]:
def classify_aqi_risk(aqi_value):
    """
    Classify an AQI value (number) into a readable risk category based on
    EPA air quality index breakpoints.

    Parameters
    ----------
    aqi_value : int or float  A numeric AQI reading.

    Returns
    -------
    str  One of: 'Good', 'Moderate', 'Unhealthy for Sensitive Groups',
         'Unhealthy', 'Very Unhealthy', 'Hazardous', or 'Invalid'
         if the value is negative.
    """
    if aqi_value < 0:
        return 'Invalid'
    elif aqi_value <= 50:
        return 'Good'
    elif aqi_value <= 100:
        return 'Moderate'
    elif aqi_value <= 150:
        return 'Unhealthy for Sensitive Groups'
    elif aqi_value <= 200:
        return 'Unhealthy'
    elif aqi_value <= 300:
        return 'Very Unhealthy'
    else:
        return 'Hazardous'
    

aqi_value = 72  #Can replace with any AQI value to test

print(classify_aqi_risk(aqi_value))

Moderate


###### This API call setup was made with the help of Claude AI in some parts to create a workable map and clarify how the API from PurpleAir was meant to be set up and used as it differs from a basic setup I did in my API weather project that I used for comparison.